In [2]:
import time
import math
import random
import json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler, autocast
from tqdm.auto import tqdm

In [3]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

In [ ]:
JSON_FILE   = ""
NPY_ROOT    = ""
CHECKPOINT  = ""
 
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE  = 32
EPOCHS      = 100
NUM_FRAMES  = 32
INPUT_SIZE  = 126          # 2 hands x 21 landmarks x 3 coords
NUM_CLASSES = 300
WORKERS     = 0
 
# SPOTER hyperparams
D_MODEL     = 256
NHEAD       = 8
NUM_LAYERS  = 6
DIM_FF      = 512
DROPOUT     = 0.1

In [5]:
class NpyKeypointDataset(Dataset):
    def __init__(self, json_file, npy_dir, split='train',
                 label_map=None, augment=False):
        with open(json_file, 'r') as f:
            wlasl_data = json.load(f)
 
        self.augment = augment
        self.samples = []
        self.action_to_idx = label_map if label_map is not None else {}
 
        for vid_id, info in wlasl_data.items():
            if info.get('subset') != split:
                continue
            npy_path = os.path.join(npy_dir, f"{vid_id}.npy")
            if not os.path.exists(npy_path):
                continue
 
            label_raw = info.get('action', info.get('label', ''))
            label_key = label_raw[0] if isinstance(label_raw, list) else label_raw
 
            if label_map is None and label_key not in self.action_to_idx:
                self.action_to_idx[label_key] = len(self.action_to_idx)
 
            self.samples.append((npy_path, label_key))
 
        print(f"[{split.upper()}] loaded {len(self.samples)} samples, "
              f"{len(self.action_to_idx)} classes")
 
    @staticmethod
    def _add_noise(x, sigma=0.01):
        return x + torch.randn_like(x) * sigma
 
    @staticmethod
    def _time_warp(x):
        T = x.shape[0]
        start = random.randint(0, T // 5)
        end   = random.randint(4 * T // 5, T)
        cropped = x[start:end].unsqueeze(0).unsqueeze(0)
        resized = nn.functional.interpolate(
            cropped, size=(T, x.shape[1]),
            mode='bilinear', align_corners=False)
        return resized.squeeze(0).squeeze(0)
 
    @staticmethod
    def _mirror_hands(x):
        x = x.clone()
        x[:, 0::3] = 1.0 - x[:, 0::3]
        return x
 
    @staticmethod
    def _scale(x, lo=0.9, hi=1.1):
        return x * random.uniform(lo, hi)
 
    def __len__(self):
        return len(self.samples)
 
    def __getitem__(self, idx):
        npy_path, label_key = self.samples[idx]
        data = np.load(npy_path, allow_pickle=True)
 
        if data.ndim == 0 and isinstance(data.item(), dict):
            features = data.item()['feature']
        else:
            features = data   # [32, 126]
 
        x = torch.FloatTensor(features)
 
        if self.augment:
            if random.random() < 0.8:
                x = self._add_noise(x)
            if random.random() < 0.7:
                x = self._time_warp(x)
            if random.random() < 0.5:
                x = self._mirror_hands(x)
            if random.random() < 0.5:
                x = self._scale(x)
 
        y = self.action_to_idx[label_key]
        return x, y
 

In [6]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float()
                        * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
 
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])
 
 
class SPOTER(nn.Module):
    def __init__(self, input_size=126, d_model=256, nhead=8,
                 num_layers=6, dim_ff=512, num_classes=300, dropout=0.1):
        super().__init__()
 
        self.input_proj = nn.Sequential(
            nn.Linear(input_size, d_model),
            nn.LayerNorm(d_model),
        )
        self.pos_enc   = PositionalEncoding(d_model, dropout=dropout)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
 
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_ff, dropout=dropout,
            batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers,
            norm=nn.LayerNorm(d_model),
        )
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes),
        )
 
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
 
        total = sum(p.numel() for p in self.parameters())
        print(f"[SPOTER] Parameters: {total:,}")
 
    def forward(self, x):
        B   = x.size(0)
        x   = self.input_proj(x)
        x   = self.pos_enc(x)
        cls = self.cls_token.expand(B, -1, -1)
        x   = torch.cat([cls, x], dim=1)
        x   = self.transformer(x)
        return self.classifier(x[:, 0])

In [7]:
def run_epoch(model, loader, optimizer, criterion, scaler,
              device, epoch, total_epochs, train=True):
    model.train() if train else model.eval()
 
    total_loss, correct, total = 0.0, 0, 0
    total_time = 0.0
    latencies  = []
 
    tag  = "Train" if train else "Val  "
    pbar = tqdm(loader, desc=f"{tag} [{epoch+1}/{total_epochs}]", leave=False)
 
    for kp, labels in pbar:
        kp, labels = kp.to(device), labels.to(device)
        t0 = time.perf_counter()
 
        if train:
            optimizer.zero_grad()
            with autocast('cuda'):
                out  = model(kp)
                loss = criterion(out, labels)
            scaler.scale(loss).backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            with torch.no_grad(), autocast('cuda'):
                out  = model(kp)
                loss = criterion(out, labels)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            t1 = time.perf_counter()
            total_time += t1 - t0
            latencies.append((t1 - t0) / kp.size(0) * 1000)
 
        total_loss += loss.item()
        _, pred = out.max(1)
        total   += labels.size(0)
        correct += pred.eq(labels).sum().item()
        pbar.set_postfix(loss=f"{loss.item():.3f}",
                         acc=f"{100.*correct/total:.1f}%")
 
    avg_loss = total_loss / len(loader)
    acc      = 100. * correct / total
    avg_lat  = float(np.mean(latencies)) if latencies else 0.0
    fps      = total / total_time if total_time > 0 else 0.0
    return avg_loss, acc, avg_lat, fps
 

In [8]:
if __name__ == "__main__":
    os.makedirs(os.path.dirname(CHECKPOINT), exist_ok=True)
 
    train_set = NpyKeypointDataset(JSON_FILE, NPY_ROOT,
                                   split='train', augment=True)
    val_set   = NpyKeypointDataset(JSON_FILE, NPY_ROOT, split='val',
                                   label_map=train_set.action_to_idx)
    test_set  = NpyKeypointDataset(JSON_FILE, NPY_ROOT, split='test',
                                   label_map=train_set.action_to_idx)
 
    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=WORKERS, pin_memory=True)
    val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=WORKERS, pin_memory=True)
    test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=WORKERS, pin_memory=True)
 
    print(f"Device : {DEVICE}")
    ACTUAL_NUM_CLASSES = len(train_set.action_to_idx)
    print(f"Classes: {ACTUAL_NUM_CLASSES}")
 
    model = SPOTER(
        input_size=INPUT_SIZE, d_model=D_MODEL, nhead=NHEAD,
        num_layers=NUM_LAYERS, dim_ff=DIM_FF,
        num_classes=ACTUAL_NUM_CLASSES, dropout=DROPOUT,
    ).to(DEVICE)
 
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
 
    def lr_lambda(epoch):
        warmup = 10
        if epoch < warmup:
            return (epoch + 1) / warmup
        progress = (epoch - warmup) / (EPOCHS - warmup)
        return 0.5 * (1 + math.cos(math.pi * progress))
 
    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler    = GradScaler('cuda')
 
    best_val_acc = 0.0
 
    for epoch in range(EPOCHS):
        t0 = time.time()
 
        train_loss, train_acc, _, _ = run_epoch(
            model, train_loader, optimizer, criterion, scaler,
            DEVICE, epoch, EPOCHS, train=True)
        val_loss, val_acc, val_lat, val_fps = run_epoch(
            model, val_loader, optimizer, criterion, scaler,
            DEVICE, epoch, EPOCHS, train=False)
 
        scheduler.step()
        duration = time.time() - t0
        lr_now   = optimizer.param_groups[0]['lr']
 
        print(f"Epoch [{epoch+1:03d}/{EPOCHS}] "
              f"lr={lr_now:.2e} | "
              f"Train {train_loss:.4f}/{train_acc:.2f}% | "
              f"Val {val_loss:.4f}/{val_acc:.2f}% | "
              f"Lat {val_lat:.2f}ms | {duration:.1f}s")
 
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc,
                'label_map': train_set.action_to_idx,
                'config': dict(input_size=INPUT_SIZE, d_model=D_MODEL,
                               nhead=NHEAD, num_layers=NUM_LAYERS,
                               dim_ff=DIM_FF, num_classes=ACTUAL_NUM_CLASSES,
                               dropout=DROPOUT),
            }, CHECKPOINT)
            print(f"  -> Best saved (val acc {val_acc:.2f}%)")
 
    # Final test
    print("\n" + "=" * 60)
    ckpt = torch.load(CHECKPOINT)
    model.load_state_dict(ckpt['model_state_dict'])
    _, test_acc, test_lat, test_fps = run_epoch(
        model, test_loader, None, criterion, scaler,
        DEVICE, 0, 1, train=False)
    print(f"Final Test Acc : {test_acc:.2f}%")
    print(f"Latency        : {test_lat:.2f} ms/video")
    print(f"FPS            : {test_fps:.1f}")

[TRAIN] loaded 1897 samples, 300 classes
[VAL] loaded 446 samples, 300 classes
[TEST] loaded 317 samples, 300 classes
Device : cuda
Classes: 300


/tmp/ipykernel_975/1823333500.py:34: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


[SPOTER] Parameters: 3,274,028


Train [1/100]:   0%|          | 0/60 [00:00<?, ?it/s]

Val   [1/100]:   0%|          | 0/14 [00:00<?, ?it/s]

...

Train [69/100]:   0%|          | 0/60 [00:00<?, ?it/s]

Val   [69/100]:   0%|          | 0/14 [00:00<?, ?it/s]

Epoch [069/100] lr=2.65e-05 | Train 4.6010/8.86% | Val 4.8666/4.93% | Lat 0.16ms | 7.0s
  -> Best saved (val acc 4.93%)


Train [70/100]:   0%|          | 0/60 [00:00<?, ?it/s]

KeyboardInterrupt: 